# Orbital Elements

## Setup

Import packages. As in the Cartesian propagation tutorial, `Munch` provides dot-notation access to dictionaries, which we use to keep related results organised.

In [1]:
import numpy as np
import astropy.units as u
from munch import Munch

import tellurion as tell

Package versions
orekit-jpype: 13.1.3.0
orekitdata: 0.dev0+untagged.82.g7294433
astropy: 7.2.0
numpy: 2.2.6


## Keplerian elements

The classical [Keplerian elements](https://en.wikipedia.org/wiki/Orbital_elements) describe an orbit by the shape and size of the ellipse (`sma`, `ecc`), its orientation in space (`inc`, `raan`, `argper`), and the position of the satellite on the ellipse at a reference epoch (`ma` or `ta`).

### Defining an orbit by semimajor axis and eccentricity

The most direct way to define a Keplerian element set is to provide all six elements explicitly. Any value supplied without units is assigned the default unit for its physical type.

In [2]:
kep = Munch()

kep.smaecc = tell.kepler(
    {"sma": 8000.0*u.km, "ecc": 0.1,
     "inc": 42.0*u.deg, "argper": 66.0*u.deg,
     "raan": 217.4*u.deg, "ma": 7.25*u.deg},
    tell.abstime('2023-09-14T08:30:00'))

kep.smaecc

ElementSetT(elements=<Quantity (8000., 0.1, 42., 66., -142.6, 7.25) (km, , deg, deg, deg, deg)>, time=<Time object: scale='utc' format='isot' value=2023-09-14T08:30:00.000>)

The result is an `ElementSetT` — an orbital element set paired with an epoch time. The `.elements` attribute is a structured `Quantity` whose fields are the element names; `.time` is an AstroPy `Time`.

In [3]:
kep.smaecc.elements.to_dict()

{'sma': <Quantity 8000. km>,
 'ecc': <Quantity 0.1>,
 'inc': <Quantity 42. deg>,
 'argper': <Quantity 66. deg>,
 'raan': <Quantity -142.6 deg>,
 'ma': <Quantity 7.25 deg>}

In [4]:
kep.smaecc.time

<Time object: scale='utc' format='isot' value=2023-09-14T08:30:00.000>

Individual elements are accessed by field name. Angles are normalised to (−180°, 180°] on construction.

In [5]:
kep.smaecc.elements['sma'], kep.smaecc.elements['ecc'], kep.smaecc.elements['inc']

(<Quantity 8000. km>, <Quantity 0.1>, <Quantity 42. deg>)

### Defining an orbit by altitude of perigee and apogee

It is often more natural to describe a low-Earth orbit by the altitudes of its perigee and apogee rather than by semimajor axis and eccentricity. The function `allplane()` converts between these representations and also computes related quantities such as mean motion and orbital radii.

In [6]:
# A near-circular LEO orbit like the ISS
kep.iss = tell.kepler(
    tell.allplane({"altper": 400*u.km, "altapo": 420*u.km,
                   "inc": 51.6*u.deg, "argper": 0.0*u.deg,
                   "raan": 0.0*u.deg, "ma": 0.0*u.deg}),
    tell.abstime('2025-01-01T00:00:00'))

kep.iss.elements.to_dict()

{'altper': <Quantity 400. km>,
 'altapo': <Quantity 420. km>,
 'inc': <Quantity 51.6 deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 0. deg>,
 'ma': <Quantity 0. deg>,
 'radper': <Quantity 6778.1365 km>,
 'radapo': <Quantity 6798.1365 km>,
 'sma': <Quantity 6788.1365 km>,
 'ecc': <Quantity 0.0015>}

`allplane()` returns a dictionary that includes all three pairs simultaneously — `(sma, ecc)`, `(radper, radapo)`, and `(altper, altapo)` — whichever way you started. We can verify the altitudes of the orbit we just defined:

In [7]:
allplane_iss = tell.allplane(kep.iss.elements.to_dict())
allplane_iss['altper'], allplane_iss['altapo'], allplane_iss['memo']

(<Quantity 400. km>, <Quantity 420. km>, <Quantity 0.0011 rad / s>)

### Defining an orbit by mean motion

For a geosynchronous orbit it is natural to specify the mean motion directly. Passing `memo` instead of `sma` to `allplane()` converts to semimajor axis automatically. The function `sma()` can also compute the semimajor axis from mean motion, orbital period, specific energy, or altitude.

In [8]:
# 1 revolution per sidereal day → geosynchronous
kep.geo = tell.kepler(
    tell.allplane({"memo": 1.0*u.rev/u.sday, "ecc": 0.0,
                   "inc": 0.0*u.deg, "argper": 0.0*u.deg,
                   "raan": 0.0*u.deg, "ma": 0.0*u.deg}),
    tell.abstime('2025-01-01T00:00:00'))

kep.geo.elements['sma'].to(u.km)

<Quantity 42164.1696 km>

The canonical GEO radius is 42 164 km; `sma()` gives the same result directly:

In [9]:
tell.sma(1.0)   # 1 rev/sday, returns semimajor axis

<Quantity 42164.1696 km>

### A GEO transfer orbit

An orbit whose apogee is at GEO altitude and whose perigee is at a low parking orbit. Here we use `sma(1.0, altitude=True)` to get the GEO altitude (i.e. GEO radius minus Earth radius) as the apogee altitude.

In [10]:
kep.gto = tell.kepler(
    tell.allplane({"altper": 350*u.km, "altapo": tell.sma(1.0, altitude=True),
                   "inc": 0.0*u.deg, "argper": 120.0*u.deg,
                   "raan": 0.0*u.deg, "ma": 90.0*u.deg}),
    tell.abstime('2025-01-01T05:55:00'))

kep.gto.elements.to_dict()

{'altper': <Quantity 350. km>,
 'altapo': <Quantity 35786.0332 km>,
 'inc': <Quantity 0. deg>,
 'argper': <Quantity 120. deg>,
 'raan': <Quantity 0. deg>,
 'ma': <Quantity 90. deg>,
 'radper': <Quantity 6728.1365 km>,
 'radapo': <Quantity 42164.1696 km>,
 'sma': <Quantity 24446.153 km>,
 'ecc': <Quantity 0.7248>}

### Reading individual element values

The function `elementval()` extracts one or more named element values from any orbital state representation, returning `Quantity` in the preferred units. It also computes derived quantities such as `altper`, `altapo`, `radper`, `radapo`, `memo`, and `period` that are not stored directly in the element set.

In [11]:
# Single element
tell.elementval(kep.gto, 'altper')

<Quantity 350. km>

In [12]:
# Several elements at once — returns a list of Quantity
tell.elementval(kep.gto, ['sma', 'ecc', 'altper', 'altapo', 'period'])

[<Quantity 24446.153 km>,
 <Quantity 0.7248>,
 <Quantity 350. km>,
 <Quantity 35786.0332 km>,
 <Quantity 38038.7967 s>]

## Converting between elements and Cartesian state

The function `pvt()` converts any element set to a Cartesian position-velocity-time (`PositionVelocityT`) object. The reverse direction is provided by the `.kepler()` method on `PositionVelocityT`.

In [13]:
# Keplerian elements → Cartesian state
pvt_iss = tell.pvt(kep.iss)
pvt_iss.cartesian

<Quantity ([6778.1365,    0.    ,    0.    ], [-0.    ,  4.7668,  6.0142]) (km, km / s)>

In [14]:
# Cartesian state → Keplerian elements
kep.iss_roundtrip = pvt_iss.kepler()
kep.iss_roundtrip.elements.to_dict()

{'ecc': <Quantity 0.0015>,
 'sma': <Quantity 6788.1365 km>,
 'inc': <Quantity 51.6 deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 0. deg>,
 'ma': <Quantity 0. deg>}

The round-trip recovers the original elements to numerical precision. We can also start from a raw Cartesian state vector, as in the Cartesian propagation tutorial:

In [15]:
# Start from a Cartesian state vector (km, km/s)
pvt_raw = tell.pvtcart(
    np.array([5740.13268349, 3314.06715, 0., -2.75082684, 4.76457184, 5.50165367]),
    tell.abstime('2025-01-01T00:00:00'))

kep.from_cart = pvt_raw.kepler()
kep.from_cart.elements.to_dict()

{'ecc': <Quantity 0.0066>,
 'sma': <Quantity 6672.3744 km>,
 'inc': <Quantity 45. deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 30. deg>,
 'ma': <Quantity -0. deg>}

The `.kepler()` method returns mean anomaly by default. To get true anomaly instead, pass `mean_time_element=False`:

In [16]:
kep.from_cart_ta = pvt_raw.kepler(mean_time_element=False)
kep.from_cart_ta.elements.to_dict()

{'ecc': <Quantity 0.0066>,
 'sma': <Quantity 6672.3744 km>,
 'inc': <Quantity 45. deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 30. deg>,
 'ta': <Quantity -0. deg>}

## Equinoctial elements

[Equinoctial elements](https://en.wikipedia.org/wiki/Equinoctial_elements) avoid the singularities that arise in Keplerian elements when eccentricity or inclination approaches zero. If the Kepler elements are defined, the equinoctial elements can be defined from them by:

| Element | Definition |
|---------|------------|
| `sma`   | Semimajor axis |
| `ex`    | $e\cos(\omega+\Omega)$ |
| `ey`    | $e\sin(\omega+\Omega)$ |
| `hx`    | $\tan(i/2)\cos\Omega$ |
| `hy`    | $\tan(i/2)\sin\Omega$ |
| `ml`    | Mean longitude $M+\omega+\Omega$ |

For a near-circular orbit `ex ≈ ey ≈ 0`, and for a zero-inclination orbit `hx ≈ hy ≈ 0` — these are exactly the cases where Keplerian elements break down.

eq = Munch()

# Convert the ISS-like orbit to equinoctial elements
pvt_iss = tell.pvt(kep.iss)
eq.iss = pvt_iss.equinoctial()
eq.iss.elements.to_dict()

Because the ISS orbit is nearly circular, `ex` and `ey` are very close to zero. For the GEO orbit, which is exactly equatorial, `hx` and `hy` should also be near zero:

In [19]:
eq.geo = tell.pvt(kep.geo).equinoctial()
eq.geo.elements.to_dict()

{'sma': <Quantity 42164.1696 km>,
 'ex': <Quantity 0.>,
 'ey': <Quantity 0.>,
 'hx': <Quantity 0.>,
 'hy': <Quantity 0.>,
 'ml': <Quantity 0. deg>}

For the GTO, which is both eccentric and inclined, all components are nonzero:

In [20]:
eq.gto = tell.pvt(kep.gto).equinoctial()
eq.gto.elements.to_dict()

{'sma': <Quantity 24446.153 km>,
 'ex': <Quantity -0.3624>,
 'ey': <Quantity 0.6277>,
 'hx': <Quantity 0.>,
 'hy': <Quantity -0.>,
 'ml': <Quantity -150. deg>}

### Reading equinoctial element values

`elementval()` dispatches automatically on the element type, so the same call works for equinoctial element sets. The semimajor axis is the same regardless of which element type is used.

In [22]:
tell.elementval(eq.iss, ['sma', 'ex', 'ey', 'hx', 'hy', 'ml'])

[<Quantity 6788.1365 km>,
 <Quantity 0.0015>,
 <Quantity 0.>,
 <Quantity 0.4834>,
 <Quantity 0.>,
 <Quantity 0. deg>]

In [24]:
# semimajor axis matches between Keplerian and equinoctial representations
tell.elementval(kep.iss, 'sma'), tell.elementval(eq.iss, 'sma')

(<Quantity 6788.1365 km>, <Quantity 6788.1365 km>)

### Round-trip: equinoctial → Cartesian → equinoctial

`pvt()` accepts equinoctial element sets in the same way as Keplerian ones.

In [25]:
pvt_from_eq = tell.pvt(eq.gto)
eq.gto_roundtrip = pvt_from_eq.equinoctial()
eq.gto_roundtrip.elements.to_dict()

{'sma': <Quantity 24446.153 km>,
 'ex': <Quantity -0.3624>,
 'ey': <Quantity 0.6277>,
 'hx': <Quantity -0.>,
 'hy': <Quantity 0.>,
 'ml': <Quantity -150. deg>}

## Circular elements

Circular elements are a hybrid set: they keep `inc` and `raan` in their familiar Keplerian form but replace `ecc` and `argper` with an eccentricity vector measured relative to the latitude argument:

| Element | Definition |
|---------|------------|
| `sma`   | Semimajor axis |
| `cex`   | $e\cos\omega$ |
| `cey`   | $e\sin\omega$ |
| `inc`   | Inclination |
| `raan`  | Right ascension of the ascending node |
| `mla`   | Mean latitude argument $M+\omega$ |

This makes circular elements a natural choice for near-circular inclined orbits such as LEO constellations — `inc` and `raan` remain directly readable, and the near-zero `cex`, `cey` no longer cause numerical issues.

In [26]:
circ = Munch()

circ.iss = pvt_iss.circular()
circ.iss.elements.to_dict()

{'sma': <Quantity 6788.1365 km>,
 'cex': <Quantity 0.0015>,
 'cey': <Quantity 0.>,
 'inc': <Quantity 51.6 deg>,
 'raan': <Quantity 0. deg>,
 'mla': <Quantity 0. deg>}

Notice that `inc` and `raan` are immediately recognisable, unlike the equinoctial `hx` and `hy`. For the ISS-like orbit, `cex` and `cey` are near zero because the orbit is nearly circular.

In [27]:
circ.gto = tell.pvt(kep.gto).circular()
circ.gto.elements.to_dict()

{'sma': <Quantity 24446.153 km>,
 'cex': <Quantity -0.3624>,
 'cey': <Quantity 0.6277>,
 'inc': <Quantity 0. deg>,
 'raan': <Quantity -0. deg>,
 'mla': <Quantity -150. deg>}

For the GTO the eccentricity components are substantial, as expected.

### Round-trip: circular → Cartesian → circular

In [28]:
pvt_from_circ = tell.pvt(circ.iss)
circ.iss_roundtrip = pvt_from_circ.circular()
circ.iss_roundtrip.elements.to_dict()

{'sma': <Quantity 6788.1365 km>,
 'cex': <Quantity 0.0015>,
 'cey': <Quantity 0.>,
 'inc': <Quantity 51.6 deg>,
 'raan': <Quantity 0. deg>,
 'mla': <Quantity 0. deg>}

## Comparing element types side by side

The same orbit can be described in all three element types simultaneously. The semimajor axis must agree across all three, and we can verify that here for the GTO.

In [29]:
pvt_gto = tell.pvt(kep.gto)

comparison = Munch()
comparison.kep  = kep.gto
comparison.eq   = pvt_gto.equinoctial()
comparison.circ = pvt_gto.circular()

print('sma (Keplerian):    ', tell.elementval(comparison.kep,  'sma').to(u.km))
print('sma (equinoctial):  ', tell.elementval(comparison.eq,   'sma').to(u.km))
print('sma (circular):     ', tell.elementval(comparison.circ, 'sma').to(u.km))

sma (Keplerian):     24446.15304164751 km
sma (equinoctial):   24446.153041647514 km
sma (circular):      24446.153041647514 km


The predicates `iskepels()`, `isequels()`, and `iscircels()` identify the type of an element set unambiguously. Exactly one will return `True` for any given `ElementSetT`.

In [30]:
print('Keplerian GTO:')
print('  iskepels:', tell.iskepels(comparison.kep))
print('  isequels:', tell.isequels(comparison.kep))
print('  iscircels:', tell.iscircels(comparison.kep))

print('Equinoctial GTO:')
print('  iskepels:', tell.iskepels(comparison.eq))
print('  isequels:', tell.isequels(comparison.eq))
print('  iscircels:', tell.iscircels(comparison.eq))

print('Circular GTO:')
print('  iskepels:', tell.iskepels(comparison.circ))
print('  isequels:', tell.isequels(comparison.circ))
print('  iscircels:', tell.iscircels(comparison.circ))

Keplerian GTO:
  iskepels: True
  isequels: False
  iscircels: False
Equinoctial GTO:
  iskepels: False
  isequels: True
  iscircels: False
Circular GTO:
  iskepels: False
  isequels: False
  iscircels: True


## Choosing an element type

| Orbit type | Recommended elements | Reason |
|---|---|---|
| General (any `ecc`, any `inc`) | Keplerian | Most familiar; no singularity in practice |
| Near-circular (`ecc` ≲ 0.01) and inclined | Circular | Keeps `inc`, `raan` readable |
| Near-equatorial (`inc` ≲ 1°) | Equinoctial | Avoids RAAN singularity |
| Near-circular **and** near-equatorial (e.g. GEO) | Equinoctial | Avoids both singularities |
| Numerical propagation (internal) | Equinoctial | Orekit's default for numerical integrators |